# KISS ICP Validation

At a chosen `TIME` this notebook:
1. Reads GPS from the rosbag and converts to Dutch RD (EPSG:28992).
2. Loads KISS ICP poses saved by `run_kiss_icp.py`.
3. Estimates bike heading from GPS displacement around `TIME`.
4. Builds a sample strip from `X_BACK` to `X_FRONT` every `DX` metres.
5. Queries AHN4 DTM once for the full strip.
6. Extracts the KISS ICP z-profile along the same strip.
7. Plots and saves both profiles.

## Settings

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt



RESULTS_DIR = Path(r"C:\Users\Leons\lidar-ground-segmentation2\1. Validation Methods\KISS ICP\KISS ICP results\Rosbag 14_30_00")
TIME = 1777638690.249665836
poses      = np.load(RESULTS_DIR / "poses.npy")       # (N, 4, 4)
timestamps = np.load(RESULTS_DIR / "timestamps.npy")   # (N,)

print(f"Loaded {len(poses)} poses from {RESULTS_DIR}/")

# =============================================================================
# PLOT
# =============================================================================

trajectory = poses[:, :3, 3]   # (N, 3)  x, y, z
x, y, z    = trajectory[:, 0], trajectory[:, 1], trajectory[:, 2]
t          = timestamps

In [ ]:
ref_idx = np.argmin(np.abs(t - TIME)) #argmin returns the index of the minimum value in the array, which corresponds to the closest timestamp to TIME
dt = abs(t[ref_idx] - TIME)
print(f"Using TIME = {TIME}, closest GPS message Δt = {dt:.3f}s " f"(index {ref_idx})")


s = np.sqrt(np.diff(x)**2 + np.diff(y)**2)
cum_s = np.concatenate([[0], np.cumsum(s)])
cum_s_from_s0 = cum_s - cum_s[ref_idx]
idx_25m = np.argmax(cum_s_from_s0 >= 20.0)
idx_5m = np.argmin(cum_s_from_s0 <= -5.0)
mask = (t >= t[idx_5m]) & (t <= t[idx_25m])
s_rel = cum_s_from_s0[mask]
z_rel = z[mask] - z[ref_idx]
t_rel = t[mask]
print(len(t), len(z))
print(s_rel)
plt.figure(figsize=(10, 5))
plt.plot(s_rel, z_rel, label="KISS ICP Z relative to 20m point")
plt.xlabel("Distance from 20m point (m)")
plt.ylabel("Z relative to 20m point (m)")   
plt.title("KISS ICP Z Position Relative to 20m Point")
plt.grid()
plt.legend()
plt.tight_layout()
plt.show()